In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array, save_img


In [ ]:

# --- Configuration ---

# 1. Directory containing your ORIGINAL blue blob images
original_blob_dir = 'galaxy_test'

# 2. Directory where you want to SAVE the AUGMENTED blue blob images
#    This directory will be created if it doesn't exist.
augmented_blob_dir = 'galaxy_test'

# 3. Number of augmented versions to create FOR EACH original image
#    Adjust this based on how much you want to increase your blob sample size.
#    If you have 30 original blobs and set this to 10, you'll get 30 * 10 = 300 augmented images.
num_augmented_per_original = 2

# 4. Target image size (should match the input size of your CNN)
IMG_WIDTH = 256
IMG_HEIGHT = 256
target_size = (IMG_HEIGHT, IMG_WIDTH)

# 5. List of image file extensions to look for
image_extensions = ('.jpeg')

# --- Augmentation Setup ---

# Create an ImageDataGenerator instance with the desired augmentations.
# These are examples, adjust them based on what makes sense for astronomical images.
# Avoid augmentations that drastically change the nature of the blobs.
datagen = ImageDataGenerator(
    rotation_range=45,       # Random rotations 
    width_shift_range=0.15,   # Random horizontal shifts 
    height_shift_range=0.15,  # Random vertical shifts 
    shear_range=0.0,         # Shearing transformation 
    zoom_range=0.1,          # Random zoom 
    horizontal_flip=True,    # Randomly flip inputs horizontally
    vertical_flip=True,      # Randomly flip inputs vertically
    fill_mode='reflect',     # How to fill newly created pixels
)

# --- Create Output Directory ---
if not os.path.exists(augmented_blob_dir):
    os.makedirs(augmented_blob_dir)
    print(f"Created directory: {augmented_blob_dir}")

# --- Augmentation Loop ---
print(f"Starting augmentation process...")
print(f"Source directory: {original_blob_dir}")
print(f"Destination directory: {augmented_blob_dir}")
print(f"Generating {num_augmented_per_original} augmented images per original image.")

total_original = 0
total_augmented = 0

# Iterate through each file in the original blue blob directory
for filename in os.listdir(original_blob_dir):
    # Check if the file is an image
    if filename.lower().endswith(image_extensions):
        total_original += 1
        original_img_path = os.path.join(original_blob_dir, filename)

        # Load the original image using Keras utilities
        # It loads the image and resizes it if necessary
        img = load_img(original_img_path, target_size=target_size)

        # Convert the image to a NumPy array with shape (height, width, channels)
        x = img_to_array(img)

        # Reshape the image to (1, height, width, channels) because the generator expects batches
        x = np.expand_dims(x, axis=0)

        print(f"  Processing: {filename}...")

        # Generate augmented images using the datagen.flow() method
        # We will loop 'num_augmented_per_original' times for each original image
        i = 0
        # The .flow() method generates batches of augmented data indefinitely.
        # We need `batch_size=1` because we're processing one image at a time here.
        # We manually break the loop after saving enough images.
        for batch in datagen.flow(x, batch_size=1):
            # Get the augmented image array from the generated batch
            augmented_img_array = batch[0]

            # Construct the new filename for the augmented image
            base_name, ext = os.path.splitext(filename)
            augmented_filename = f"{base_name}_aug_{i}{ext}"
            augmented_img_path = os.path.join(augmented_blob_dir, augmented_filename)

            # Save the augmented image array to a file
            save_img(augmented_img_path, augmented_img_array)

            i += 1
            total_augmented += 1
            # Break the loop once we have generated the desired number of augmented images
            if i >= num_augmented_per_original:
                break

print("-" * 30)
print("Offline augmentation complete!")
print(f"Processed {total_original} original blue blob images.")
print(f"Generated {total_augmented} augmented images in {augmented_blob_dir}")
print("-" * 30)
print("Next Steps:")
print("1. Verify some augmented images visually to ensure they look reasonable.")
print("2. Create your training dataset by combining:")
print("   - Your original galaxy images")
print("   - Your original blue blob images")
print("   - These newly generated augmented blue blob images")
print("3. Train your model on this expanded dataset. You might still want to use class weights,")
print("   or adjust them now that the classes are less imbalanced.")

Starting augmentation process...
Source directory: Downsample_stat_rot_images
Destination directory: Augmented_images
Generating 2 augmented images per original image.
  Processing: _BC6_aug_003.jpeg...
  Processing: BC25_aug_001.jpeg...
  Processing: _BC9_aug_007.jpeg...
  Processing: _BC1_aug_003.jpeg...
  Processing: BC26_aug_003.jpeg...
  Processing: _BC5_aug_001.jpeg...
  Processing: BC22_aug_001.jpeg...
  Processing: BC29_aug_007.jpeg...
  Processing: CCO1_aug_007.jpeg...
  Processing: BC37_aug_001.jpeg...
  Processing: BC33_aug_003.jpeg...
  Processing: BC38_aug_005.jpeg...
  Processing: BC30_aug_001.jpeg...
  Processing: BC34_aug_003.jpeg...
  Processing: BC11_aug_002.jpeg...
  Processing: BC39_aug_008.jpeg...
  Processing: BC40_aug_000.jpeg...
  Processing: BC15_aug_000.jpeg...
  Processing: BC43_aug_002.jpeg...
  Processing: BC16_aug_002.jpeg...
  Processing: BC47_aug_000.jpeg...
  Processing: BC12_aug_000.jpeg...
  Processing: BC20_aug_002.jpeg...
  Processing: _BC7_aug_002.

In [5]:
import os
len(os.listdir("galaxies_test_images_subset")), len(os.listdir("galaxy_train")), len(os.listdir("galaxy_val"))

(354, 680, 306)